# 7.18 Semantic segmentation (FCN, U-Net)

Semantic segmentation is classification without looking away from the grid: every pixel receives a class, so boundaries become first-class predictions.

FCNs keep spatial layout for dense labels, and U-Net skip connections restore detail when coarse semantic maps are upsampled.

Save a copy to Drive to edit.

In [ ]:
import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)
rng = np.random.default_rng(7)

## Build pixel metrics and skip concatenation

At every pixel, $p_{u,v,c}=e^{z_{u,v,c}}/\sum_k e^{z_{u,v,k}}$ and $L=-\log p_{u,v,y}$. We verify the exact 3x3 mask, class IoU, skip shape, and cross-entropy.

In [ ]:
def class_iou(pred, true, cls):
    pred_mask = pred == cls
    true_mask = true == cls
    inter = np.logical_and(pred_mask, true_mask).sum()
    union = np.logical_or(pred_mask, true_mask).sum()
    if union == 0:
        return 1.0
    return float(inter / union)


def mean_foreground_iou(pred, true):
    labels = sorted(set(np.unique(true).tolist()) - {0})
    values = [class_iou(pred, true, cls) for cls in labels]
    return float(np.mean(values))


def nearest_resize(mask, factor):
    return np.repeat(np.repeat(mask, factor, axis=0), factor, axis=1)


def make_mask(size, boxes):
    mask = np.zeros((size, size), dtype=int)
    for cls, box in boxes:
        x1, y1, x2, y2 = box
        mask[y1:y2, x1:x2] = cls
    return mask


def shift_mask(mask, amount):
    shifted = np.roll(mask, amount, axis=1)
    shifted[:, :amount] = 0
    return shifted


def load_segmentation_ladder():
    truth = []
    pred = []
    truth.append(np.array([[0, 0, 1], [0, 1, 1], [2, 2, 1]]))
    pred.append(np.array([[0, 0, 1], [0, 1, 0], [2, 2, 1]]))
    truth.append(make_mask(8, [(1, [1, 1, 5, 5]), (2, [5, 4, 7, 7])]))
    pred.append(shift_mask(truth[-1], 1))
    truth.append(make_mask(12, [(1, [1, 1, 7, 8]), (2, [7, 3, 11, 10]), (3, [3, 8, 8, 11])]))
    pred.append(shift_mask(truth[-1], 1))
    truth.append(make_mask(18, [(1, [1, 2, 8, 11]), (2, [7, 5, 15, 14]), (3, [2, 12, 10, 17]), (1, [12, 1, 17, 6])]))
    pred.append(shift_mask(truth[-1], 2))
    truth.append(make_mask(24, [(1, [1, 1, 8, 10]), (2, [7, 4, 16, 15]), (3, [15, 2, 23, 9]), (1, [3, 15, 11, 23]), (2, [13, 14, 23, 23])]))
    noisy = shift_mask(truth[-1], 2)
    noisy[0:5, 0:8] = 0
    pred.append(noisy)
    names = ["D1 3x3 hand mask", "D2 clean parts", "D3 touching regions", "D4 thin boundaries", "D5 crowded noisy mask"]
    return list(zip(names, truth, pred))

def segment_pixels():
    true = np.array([[0, 0, 1], [0, 1, 1], [2, 2, 1]])
    pred = np.array([[0, 0, 1], [0, 1, 0], [2, 2, 1]])
    counts = np.bincount(true.ravel(), minlength=3)
    accuracy = float(np.mean(pred == true))
    iou_one = class_iou(pred, true, 1)
    encoder = np.zeros((2, 2, 3))
    decoder = np.zeros((2, 2, 2))
    skip = np.concatenate([encoder, decoder], axis=2)
    logits = np.array([1.0, 2.0, 0.0])
    probs = np.exp(logits) / np.exp(logits).sum()
    ce = float(-np.log(probs[1]))
    return counts, accuracy, iou_one, skip.shape, probs[1], ce

counts, accuracy, iou_one, skip_shape, p_true, ce = segment_pixels()
print(counts.tolist(), round(accuracy, 3), iou_one, skip_shape, round(p_true, 3), round(ce, 3))
assert counts.tolist() == [3, 4, 2]
assert round(accuracy, 3) == 0.889
assert iou_one == 0.75
assert skip_shape == (2, 2, 5)
assert round(ce, 3) == 0.408

The ladder uses synthetic masks instead of classification images: D1 is the lesson's 3x3 grid, and D5 adds crowded classes, boundaries, and missing regions.

In [ ]:
rungs = load_segmentation_ladder()
for name, true, pred in rungs:
    print(name, true.shape, "labels", sorted(np.unique(true).tolist()))

fig, axes = plt.subplots(2, 5, figsize=(14, 5))
for idx, (name, true, pred) in enumerate(rungs):
    axes[0, idx].imshow(true, vmin=0, vmax=3)
    axes[0, idx].set_title(name.split()[0] + " truth")
    axes[1, idx].imshow(pred, vmin=0, vmax=3)
    axes[1, idx].set_title(name.split()[0] + " pred")
    axes[0, idx].set_xticks([])
    axes[0, idx].set_yticks([])
    axes[1, idx].set_xticks([])
    axes[1, idx].set_yticks([])
plt.tight_layout()
plt.show()

## Run the same method across D1–D5

The metric is foreground mean IoU, which is stricter than pixel accuracy when background dominates.

In [ ]:
results = []
for name, true, pred in rungs:
    acc = float(np.mean(pred == true))
    miou = mean_foreground_iou(pred, true)
    results.append((name, acc, miou))

print("rung                         pixel_acc  mean_iou")
for name, acc, miou in results:
    print(f"{name:28s} {acc:.3f}      {miou:.3f}")

## Results visualization

The small multiples compare true and predicted masks. The curve shows foreground IoU as masks become more crowded and noisy.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(14, 8))
for idx, (name, true, pred) in enumerate(rungs):
    axes[0, idx].imshow(true, vmin=0, vmax=3)
    axes[0, idx].set_title(name.split()[0] + " truth")
    axes[1, idx].imshow(pred, vmin=0, vmax=3)
    axes[1, idx].set_title(name.split()[0] + " pred")
    axes[0, idx].set_xticks([])
    axes[0, idx].set_yticks([])
    axes[1, idx].set_xticks([])
    axes[1, idx].set_yticks([])

xs = np.arange(1, 6)
ys = [miou for name, acc, miou in results]
axes[2, 0].plot(xs, ys, marker="o")
axes[2, 0].set_xticks(xs)
axes[2, 0].set_ylim(0.0, 1.05)
axes[2, 0].set_xlabel("complexity rung")
axes[2, 0].set_ylabel("foreground IoU")
axes[2, 0].grid(True, alpha=0.3)
for ax in axes[2, 1:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Pitfall on D5

Reporting only pixel accuracy can hide a missed foreground region, and resizing masks like photos creates invalid fractional labels. Use class IoU and nearest-neighbor masks.

In [ ]:
name, true, pred = rungs[-1]
background_only = np.zeros_like(true)
background_acc = float(np.mean(background_only == true))
background_iou = mean_foreground_iou(background_only, true)
nearest = nearest_resize(true, 2)
photo_like = np.kron(true.astype(float), np.ones((2, 2)))
photo_like[1::2, 1::2] = photo_like[1::2, 1::2] * 0.5
print("background-only pixel accuracy", round(background_acc, 3))
print("background-only foreground IoU", round(background_iou, 3))
print("nearest labels", sorted(np.unique(nearest).astype(int).tolist()))
print("photo-like fractional label sample", float(photo_like[1, 1]))

## Evaluate it + Practice

- Compare the reported foreground IoU with a no-skill baseline that predicts one large center box or all background.
- Overfit D1: the hand scene should reproduce the exact lesson arithmetic before scaling up.
- Ablate the key idea, such as sorting before NMS, refinement, objectness, focal weighting, matching, or nearest-neighbor masks.
- Watch for failure signals: duplicate boxes, high pixel accuracy with poor IoU, and metrics that improve only because the scene got easier.

Practice:
1. Change the D3 object positions and predict how foreground IoU changes.
2. Tighten the matching threshold and rerun the table.
3. Add one noisy false positive to D5 and explain the curve.